# A* Search Algorithm — 8-Puzzle

**Implementation of the A\* pseudocode from Lecture 3B, Slide 21**

This notebook implements A\* **exactly** following the 8-step pseudocode:

1. Put the start node `s` on a list called **OPEN**, and compute `f(s)`.
2. If OPEN is empty, exit with failure; otherwise continue.
3. Remove from OPEN the node whose `f` value is smallest and put it on a list called **CLOSED**. Call this node `n`. (Ties for minimal `f` are resolved arbitrarily, but always in favor of a goal node.)
4. If `n` is a goal node, exit successfully with the solution obtained by tracing back the pointers; otherwise continue.
5. Expand node `n`, generating all of its successors. If `n` has no successors, go immediately to step 2.
6. For each successor `n_i` not already on either OPEN or CLOSED: compute `f(n_i)`. Put these nodes on OPEN and direct pointers from them back to `n`.
7. For successors already on OPEN or CLOSED: compare the `f` value just computed with the previous `f` value. Keep the smaller `f`. Put on OPEN those successors on CLOSED whose `f` values were lowered, and redirect to `n` the pointers from all nodes whose `f` values were lowered.
8. Go to step 2.

**Comments honored in this implementation:**
- Duplicates are not retained — a state that is regenerated updates the *existing* node instead of creating a new one.
- When a state already on OPEN or CLOSED is rediscovered via a cheaper path, the ancestor (parent pointer) is updated, and if it was on CLOSED it is moved back onto OPEN.

We implement three heuristic functions:
- **h1** — Number of tiles in the wrong position.
- **h2** — Sum of Manhattan (city-block) distances of every tile from its goal position.
- **h3** — Nilsson's Sequence Score: `h(n) = P(n) + 3*S(n)`, where `P(n)` is the Manhattan distance sum and `S(n)` is the sequence score (2 points for every non-central tile not followed by its proper clockwise successor, 0 otherwise, plus 1 point if the center square is occupied by a non-blank tile).

## 1. Puzzle representation

The 3×3 board is represented as a flat tuple of length 9, in row-major order, with `0` representing the blank tile.

In [1]:
def to_grid(state):
    """Convert a flat 9-tuple into a list of 3 rows for display."""
    return [state[0:3], state[3:6], state[6:9]]


def format_state(state):
    """One-line human readable representation, e.g. '2 1 6 / 4 * 8 / 7 5 3'."""
    rows = to_grid(state)
    return ' / '.join(' '.join('*' if x == 0 else str(x) for x in row) for row in rows)


def print_state(state):
    for row in to_grid(state):
        print(' '.join('*' if x == 0 else str(x) for x in row))
    print()


def get_successors(state):
    """Return all states reachable by sliding a tile into the blank."""
    idx = state.index(0)
    r, c = divmod(idx, 3)
    moves = []
    if r > 0: moves.append(-3)   # blank moves up
    if r < 2: moves.append(3)    # blank moves down
    if c > 0: moves.append(-1)   # blank moves left
    if c < 2: moves.append(1)    # blank moves right

    successors = []
    for m in moves:
        new_idx = idx + m
        new_state = list(state)
        new_state[idx], new_state[new_idx] = new_state[new_idx], new_state[idx]
        successors.append(tuple(new_state))
    return successors


## 2. Reading the puzzle from an input file

Instead of hard-coding the puzzle, we read it from a text file with the following format
(blank tile written as `*`):

```
start
2 1 6
4 * 8
7 5 3
goal
1 2 3
8 * 4
7 6 5
```

Put this file (e.g. named `puzzle_input.txt`) in the same folder as this notebook.

In [2]:
def parse_puzzle_file(filepath):
    """
    Parse a puzzle input file of the form:

        start
        2 1 6
        4 * 8
        7 5 3
        goal
        1 2 3
        8 * 4
        7 6 5

    '*' (or '0') marks the blank tile. Returns (start_state, goal_state) as
    flat 9-tuples in row-major order, with 0 for the blank.
    """
    with open(filepath) as f:
        lines = [line.strip() for line in f if line.strip()]

    def find_label(label):
        for i, line in enumerate(lines):
            if line.lower() == label:
                return i
        raise ValueError(f"Could not find '{label}' section in {filepath}")

    def parse_grid(grid_lines):
        state = []
        for line in grid_lines:
            for tok in line.split():
                state.append(0 if tok in ('*', '0') else int(tok))
        if len(state) != 9:
            raise ValueError(f"Expected 9 tile values, got {len(state)}: {state}")
        return tuple(state)

    start_idx = find_label('start')
    goal_idx = find_label('goal')

    start_grid_lines = lines[start_idx + 1: start_idx + 4]
    goal_grid_lines = lines[goal_idx + 1: goal_idx + 4]

    return parse_grid(start_grid_lines), parse_grid(goal_grid_lines)


In [3]:
PUZZLE_FILE = "puzzle_input.txt"   # change this if your file has a different name/path

START, GOAL = parse_puzzle_file(PUZZLE_FILE)

print("START:")
print_state(START)
print("GOAL:")
print_state(GOAL)


START:
2 1 6
4 * 8
7 5 3

GOAL:
1 2 3
8 * 4
7 6 5



## 3. Heuristic functions

In [4]:
def h_wrong_tiles(state, goal=GOAL):
    """h1: number of tiles (excluding the blank) not in their goal position."""
    return sum(1 for i in range(9) if state[i] != 0 and state[i] != goal[i])


def h_manhattan(state, goal=GOAL):
    """h2: sum of Manhattan distances of every tile (excluding blank) from its goal cell."""
    total = 0
    for i in range(9):
        val = state[i]
        if val == 0:
            continue
        gi = goal.index(val)
        r1, c1 = divmod(i, 3)
        r2, c2 = divmod(gi, 3)
        total += abs(r1 - r2) + abs(c1 - c2)
    return total


### Nilsson's Sequence Score

`h(n) = P(n) + 3 * S(n)`

- `P(n)` is the Manhattan distance (h2 above).
- `S(n)` is obtained by going clockwise around the 8 **non-central** squares. For each square, look at the tile that should follow it (according to the goal's clockwise cycle `1 -> 2 -> 3 -> 4 -> 5 -> 6 -> 7 -> 8 -> 1`). If the actual next tile (clockwise) is *not* that proper successor, add 2; otherwise add 0. Finally, if the center square is occupied by a (non-blank) tile, add 1.

In [5]:
# Clockwise order of the 8 non-central (perimeter) squares, by flat index:
#   0 1 2
#   3 . 5
#   6 7 8
PERIMETER = [0, 1, 2, 5, 8, 7, 6, 3]
CENTER = 4


def build_successor_map(goal):
    """Build the cyclic 'proper successor' relation from the goal's clockwise arrangement."""
    goal_vals_in_order = [goal[p] for p in PERIMETER]
    n = len(goal_vals_in_order)
    succ = {}
    for idx, v in enumerate(goal_vals_in_order):
        succ[v] = goal_vals_in_order[(idx + 1) % n]
    return succ


SUCCESSOR_MAP = build_successor_map(GOAL)
print("Proper-successor cycle (clockwise):", SUCCESSOR_MAP)


def nilsson_S(state, goal=GOAL):
    s = 0
    n = len(PERIMETER)
    for idx in range(n):
        cur_val = state[PERIMETER[idx]]
        next_val = state[PERIMETER[(idx + 1) % n]]
        expected = SUCCESSOR_MAP.get(cur_val)
        if expected != next_val:
            s += 2
    if state[CENTER] != 0:
        s += 1
    return s


def h_nilsson(state, goal=GOAL):
    """h3: Nilsson's Sequence Score. Returns (h, P, S)."""
    P = h_manhattan(state, goal)
    S = nilsson_S(state, goal)
    return P + 3 * S, P, S


Proper-successor cycle (clockwise): {1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 8, 8: 1}


## 4. A* search — following the slide-21 pseudocode step by step

In [6]:
def compute_h(state, goal, heuristic_name):
    """Dispatch to the requested heuristic. Returns (h, P, S); P and S are None
    unless heuristic_name == 'nilsson'."""
    if heuristic_name == 'wrong':
        return h_wrong_tiles(state, goal), None, None
    elif heuristic_name == 'manhattan':
        return h_manhattan(state, goal), None, None
    elif heuristic_name == 'nilsson':
        h, P, S = h_nilsson(state, goal)
        return h, P, S
    else:
        raise ValueError("unknown heuristic: " + heuristic_name)


def a_star(start, goal, heuristic_name):
    """
    A* search implemented following the 8-step pseudocode (Lecture 3B, slide 21).

    nodes[state] holds: g, h, f, parent, status ('OPEN'/'CLOSED'), and P, S (Nilsson only).
    OPEN is kept as an ordered list (insertion order) so that ties for minimal f
    can be broken 'arbitrarily' (first inserted), except that a goal node is
    always preferred when tied for minimal f (per step 3).

    Returns: (path, nodes, nodes_generated)
        path -> list of states from start to goal (None if no solution)
        nodes -> dict of all node records (used to report g, h, f, P, S per step)
        nodes_generated -> total number of distinct nodes generated during the search
    """
    nodes = {}
    OPEN = []
    nodes_generated = 0

    # ---- Step 1: put s on OPEN, compute f(s) ----
    h0, P0, S0 = compute_h(start, goal, heuristic_name)
    nodes[start] = {'g': 0, 'h': h0, 'f': 0 + h0, 'parent': None,
                     'status': 'OPEN', 'P': P0, 'S': S0}
    OPEN.append(start)
    nodes_generated += 1

    while True:
        # ---- Step 2: if OPEN empty, exit failure ----
        if not OPEN:
            return None, nodes, nodes_generated

        # ---- Step 3: remove node with smallest f from OPEN, put on CLOSED ----
        min_f = min(nodes[s]['f'] for s in OPEN)
        candidates = [s for s in OPEN if nodes[s]['f'] == min_f]
        n = goal if goal in candidates else candidates[0]   # tie -> favor goal node
        OPEN.remove(n)
        nodes[n]['status'] = 'CLOSED'

        # ---- Step 4: if n is goal, exit success (trace back pointers) ----
        if n == goal:
            path = []
            cur = n
            while cur is not None:
                path.append(cur)
                cur = nodes[cur]['parent']
            path.reverse()
            return path, nodes, nodes_generated

        # ---- Step 5: expand n; if no successors, go to step 2 ----
        successors = get_successors(n)
        if not successors:
            continue

        g_n = nodes[n]['g']
        for s in successors:
            g_new = g_n + 1

            if s not in nodes:
                # ---- Step 6: brand-new successor -> compute f, put on OPEN ----
                h_new, P_new, S_new = compute_h(s, goal, heuristic_name)
                nodes[s] = {'g': g_new, 'h': h_new, 'f': g_new + h_new, 'parent': n,
                            'status': 'OPEN', 'P': P_new, 'S': S_new}
                OPEN.append(s)
                nodes_generated += 1
            else:
                # ---- Step 7: successor already on OPEN or CLOSED ----
                f_new = g_new + nodes[s]['h']
                if f_new < nodes[s]['f']:
                    was_closed = (nodes[s]['status'] == 'CLOSED')
                    nodes[s]['g'] = g_new
                    nodes[s]['f'] = f_new
                    nodes[s]['parent'] = n
                    if was_closed:
                        nodes[s]['status'] = 'OPEN'
                        OPEN.append(s)
                # else: existing f is already smaller/equal -> keep it, do nothing
        # ---- Step 8: go to step 2 ----


## 5. Helper to print results in the required format

For every step from start to goal we print the state, `g(n)`, `h(n)`, `f(n)` and, for Nilsson's heuristic, `P(n)` and `S(n)`. The search cost (number of nodes generated) is printed at the end.

In [7]:
def run_and_report(start, goal, heuristic_name, title):
    print("=" * 70)
    print(title)
    print("=" * 70)

    path, nodes, nodes_generated = a_star(start, goal, heuristic_name)

    if path is None:
        print("No solution found.")
        print("Search cost (nodes generated):", nodes_generated)
        print()
        return

    header = f"{'Step':<6}{'State':<24}{'g(n)':<8}{'h(n)':<8}{'f(n)':<8}"
    if heuristic_name == 'nilsson':
        header += f"{'P(n)':<8}{'S(n)':<8}"
    print(header)
    print("-" * len(header))

    for i, state in enumerate(path):
        info = nodes[state]
        row = f"{i:<6}{format_state(state):<24}{info['g']:<8}{info['h']:<8}{info['f']:<8}"
        if heuristic_name == 'nilsson':
            row += f"{info['P']:<8}{info['S']:<8}"
        print(row)

    print()
    print(f"Solution length (number of moves): {len(path) - 1}")
    print(f"Search cost (number of nodes generated): {nodes_generated}")
    print()
    return path, nodes, nodes_generated


## 6a. Heuristic (a): Number of Tiles in the Wrong Position

In [8]:
result_wrong = run_and_report(START, GOAL, 'wrong', "Heuristic (a): Number of Tiles in the Wrong Position")

Heuristic (a): Number of Tiles in the Wrong Position


Step  State                   g(n)    h(n)    f(n)    
------------------------------------------------------
0     2 1 6 / 4 * 8 / 7 5 3   0       7       7       
1     2 1 6 / 4 8 * / 7 5 3   1       7       8       
2     2 1 * / 4 8 6 / 7 5 3   2       7       9       
3     2 * 1 / 4 8 6 / 7 5 3   3       7       10      
4     2 8 1 / 4 * 6 / 7 5 3   4       7       11      
5     2 8 1 / 4 6 * / 7 5 3   5       7       12      
6     2 8 1 / 4 6 3 / 7 5 *   6       7       13      
7     2 8 1 / 4 6 3 / 7 * 5   7       6       13      
8     2 8 1 / 4 * 3 / 7 6 5   8       5       13      
9     2 8 1 / * 4 3 / 7 6 5   9       5       14      
10    * 8 1 / 2 4 3 / 7 6 5   10      5       15      
11    8 * 1 / 2 4 3 / 7 6 5   11      5       16      
12    8 1 * / 2 4 3 / 7 6 5   12      5       17      
13    8 1 3 / 2 4 * / 7 6 5   13      4       17      
14    8 1 3 / 2 * 4 / 7 6 5   14      3       17      
15    8 1 3 / * 2 4 / 7 6 5   15      3       18      
16    * 1 

## 6b. Heuristic (b): Manhattan Distance

In [9]:
result_manhattan = run_and_report(START, GOAL, 'manhattan', "Heuristic (b): Manhattan Distance")

Heuristic (b): Manhattan Distance
Step  State                   g(n)    h(n)    f(n)    
------------------------------------------------------
0     2 1 6 / 4 * 8 / 7 5 3   0       12      12      
1     2 1 6 / 4 8 * / 7 5 3   1       11      12      
2     2 1 * / 4 8 6 / 7 5 3   2       10      12      
3     2 * 1 / 4 8 6 / 7 5 3   3       11      14      
4     2 8 1 / 4 * 6 / 7 5 3   4       12      16      
5     2 8 1 / 4 6 * / 7 5 3   5       11      16      
6     2 8 1 / 4 6 3 / 7 5 *   6       10      16      
7     2 8 1 / 4 6 3 / 7 * 5   7       9       16      
8     2 8 1 / 4 * 3 / 7 6 5   8       8       16      
9     2 8 1 / * 4 3 / 7 6 5   9       7       16      
10    * 8 1 / 2 4 3 / 7 6 5   10      8       18      
11    8 * 1 / 2 4 3 / 7 6 5   11      7       18      
12    8 1 * / 2 4 3 / 7 6 5   12      6       18      
13    8 1 3 / 2 4 * / 7 6 5   13      5       18      
14    8 1 3 / 2 * 4 / 7 6 5   14      4       18      
15    8 1 3 / * 2 4 / 7 6 5   1

## 6c. Heuristic (c): Nilsson's Sequence Score

In [10]:
result_nilsson = run_and_report(START, GOAL, 'nilsson', "Heuristic (c): Nilsson's Sequence Score")

Heuristic (c): Nilsson's Sequence Score
Step  State                   g(n)    h(n)    f(n)    P(n)    S(n)    
----------------------------------------------------------------------
0     2 1 6 / 4 * 8 / 7 5 3   0       60      60      12      16      
1     2 1 6 / 4 8 * / 7 5 3   1       62      63      11      17      
2     2 1 * / 4 8 6 / 7 5 3   2       61      63      10      17      
3     2 * 1 / 4 8 6 / 7 5 3   3       62      65      11      17      
4     2 8 1 / 4 * 6 / 7 5 3   4       54      58      12      14      
5     2 8 1 / 4 6 * / 7 5 3   5       56      61      11      15      
6     2 8 1 / 4 6 3 / 7 5 *   6       55      61      10      15      
7     2 8 1 / 4 6 3 / 7 * 5   7       54      61      9       15      
8     2 8 1 / 4 * 3 / 7 6 5   8       38      46      8       10      
9     2 8 1 / * 4 3 / 7 6 5   9       40      49      7       11      
10    * 8 1 / 2 4 3 / 7 6 5   10      41      51      8       11      
11    8 * 1 / 2 4 3 / 7 6 5   11     

## 7. Summary comparison

In [11]:
print(f"{'Heuristic':<40}{'Solution length':<18}{'Nodes generated':<18}")
print("-" * 76)
for label, result in [
    ("(a) Tiles in wrong position", result_wrong),
    ("(b) Manhattan distance", result_manhattan),
    ("(c) Nilsson's Sequence Score", result_nilsson),
]:
    path, nodes, nodes_generated = result
    print(f"{label:<40}{len(path)-1:<18}{nodes_generated:<18}")


Heuristic                               Solution length   Nodes generated   
----------------------------------------------------------------------------
(a) Tiles in wrong position             18                2844              
(b) Manhattan distance                  18                376               
(c) Nilsson's Sequence Score            18                65                
